<a href="https://colab.research.google.com/github/kundrapuharsha/ai-mentor-portfolio/blob/main/day9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langgraph langchain-google-genai langchain-community duckduckgo-search


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [ ]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""
    return DuckDuckGoSearchRun().run(query)

# Test the tool directly
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

How do you create remarkable change? By hiring, celebrating and nurturing the best people-from all walks of life. We’re here to help! Tell us what you’re looking for and we’ll get you connected to the right people. Discover the latest TCS hiring 2026 updates, including TCS recruitment 2026, upcoming TCS jobs 2026, and all new TCS vacancies 2026 across India. Find complete details on TCS freshers h


In [ ]:
pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 10.0 MB/s eta 0:00:00


In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
agent = create_react_agent(llm, tools=[web_search])

print('Agent created.')

Agent created.


/tmp/ipykernel_495/1070485237.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[web_search])


In [ ]:
result = agent.invoke({
    'messages': [('user', "What is TCS's 2026 hiring quota?")]
})

# Print every message in the conversation
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f'    Tool calls: {m.tool_calls}')


[0] HumanMessage
    Content: What is TCS's 2026 hiring quota?

[1] AIMessage
    Content: 
    Tool calls: [{'name': 'web_search', 'args': {'query': 'TCS 2026 hiring quota'}, 'id': 'a0d45168-3d33-43ca-89c8-46559ed0f2e6', 'type': 'tool_call'}]

[2] ToolMessage
    Content: Feb 24, 2026 ... FREE quota exhausted! Serious. ₹299 ₹49. Browse all Jobs; Apply to all ... TCS-NQT-Hiring-2025-2026. Prerna just got referred for a SDE2 position in ... Feb 19, 2026 ... Just one exam and your life changes forever. No more struggling with a 3 LPA salary. TCS has officially announced

[3] AIMessage
    Content: [{'type': 'text', 'text': "I couldn't find a specific hiring quota for TCS in 2026. However, some sources indicate that TCS might be cutting 12,000 jobs in 2025-26, with mid and senior management being targeted due to skill mismatch. There's also mention of a TCS-NQT-Hiring-2025-2026 drive, but no s


In [ ]:
result = agent.invoke({
    'messages': [('user', 'Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd')]
})

# Watch how the agent recovers
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    {str(m.content)[:300]}')


[0] HumanMessage
    Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd

[1] AIMessage
    

[2] ToolMessage
    ERROR: failed to fetch URL — HTTPSConnectionPool(host='this-domain-does-not-exist-12345.example.com', port=443): Max retries exceeded with url: /jd (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7acb88dacda0>: Failed to resolve 'this-domain-does-not-exist-12345.examp

[3] AIMessage
    I am sorry, but I was unable to retrieve content from the URL you provided. The domain does not seem to exist.


In [ ]:
import requests
from bs4 import BeautifulSoup
from langchain_core.tools import tool

@tool
def jd_fetcher(url: str) -> str:
    """Fetch a job description from a URL and return clean plain text.
    Use when the user provides a job posting URL and you need the JD content.
    Returns first 4000 characters of the cleaned page text."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:4000]
    except Exception as e:
        return f'ERROR: failed to fetch URL — {e}'

In [ ]:
@tool
def skills_gap(student_skills: str, must_have_skills: str) -> str:
    """Compare a student's skills (comma-separated) to a job's must-have skills (comma-separated).
    Returns missing skills, comma-separated, or 'none' if student has all.
    Use when the user provides a student profile and a JD's required skills."""
    a = set(s.strip().lower() for s in student_skills.split(',') if s.strip())
    b = set(s.strip().lower() for s in must_have_skills.split(',') if s.strip())
    missing = sorted(b - a)
    return ', '.join(missing) if missing else 'none'

# Test
print(skills_gap.invoke({
    'student_skills': 'Python, Java, SQL',
    'must_have_skills': 'Python, Java, SQL, Spring Boot, AWS',
}))

aws, spring boot


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

@tool
def answer_scorer(question: str, answer: str) -> str:
    """Score a student's answer to a placement interview question, 1-10, with one-line rationale.
    Use when evaluating how well a student answered a specific interview question.
    Returns format: 'Score: X/10. Rationale: <reason>'."""
    prompt = (f'Score this placement interview answer 1-10 with one-line rationale.\n'
              f'Question: {question}\n'
              f'Answer: {answer}')
    return llm.invoke(prompt).content

# Test
print(answer_scorer.invoke({
    'question': 'Why TCS Digital?',
    'answer': 'Because TCS is big and they pay well.',
}))

**2/10**

**Rationale:** Generic and transactional, showing no specific research or interest in TCS *Digital*'s work or mission.


In [ ]:
from langgraph.prebuilt import create_react_agent

tools = [jd_fetcher, skills_gap, answer_scorer]
agent = create_react_agent(llm, tools=tools)
print(f'Agent created with {len(tools)} tools.')

Agent created with 3 tools.


/tmp/ipykernel_495/3943891205.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=tools)


In [26]:
import json
import pathlib

raw_profiles_data = json.loads(
    pathlib.Path('/content/student_profiles.json').read_text()
)

# The JSON already contains a list of profiles
profiles = raw_profiles_data

for i, p in enumerate(profiles):
    print(f'\n{"="*70}')
    print(f'Student {i+1}: {p["name"]} — {p["branch"]} CGPA {p["cgpa"]} → {p["target_company"]}')
    print(f'{"="*70}')

    msg = (
        f"I am {p['name']}, B.Tech {p['branch']} CGPA {p['cgpa']}, "
        f"skills: {', '.join(p['skills'])}. "
        f"Target: {p['target_company']}. "
        f"Plan 3 mock interview questions for me, score one of my sample answers, "
        f"and tell me what skills I need to add to be a strong fit."
    )

    result = agent.invoke(
        {'messages': [('user', msg)]},
        config={'recursion_limit': 10}
    )

    for j, m in enumerate(result['messages']):
        print(f'\n  [{j}] {type(m).__name__}')

        if hasattr(m, 'content') and m.content:
            print(f'      {str(m.content)[:300]}')

        if hasattr(m, 'tool_calls') and m.tool_calls:
            for tc in m.tool_calls:
                print(f'      → tool_call: {tc.get("name")}({tc.get("args")})')


Student 1: Rahul Kumar — Computer Science and Engineering CGPA 8.7 → Google

  [0] HumanMessage
      I am Rahul Kumar, B.Tech Computer Science and Engineering CGPA 8.7, skills: Python, Java, React, Node.js, MongoDB, Git. Target: Google. Plan 3 mock interview questions for me, score one of my sample answers, and tell me what skills I need to add to be a strong fit.

  [1] AIMessage
      [{'type': 'text', 'text': 'Hello Rahul, great to hear about your profile and your ambition to join Google!\n\nI can certainly help you with scoring a sample answer and identifying skill gaps, but I\'ll need a bit more information.\n\n1.  **Mock Interview Questions**: I can\'t generate mock interview

Student 2: Sneha Reddy — Information Technology CGPA 9.1 → Amazon

  [0] HumanMessage
      I am Sneha Reddy, B.Tech Information Technology CGPA 9.1, skills: Java, Spring Boot, MySQL, AWS, Docker. Target: Amazon. Plan 3 mock interview questions for me, score one of my sample answers, and tell me what skil

In [27]:
# Pass a bad URL — see how agent recovers
result = agent.invoke({
    'messages': [('user', 'Fetch this JD and tell me the must-have skills: '
                          'https://this-does-not-exist-99999.example.com/jd')]
}, config={'recursion_limit': 5})

print('Failure recovery trace:')
for j, m in enumerate(result['messages']):
    print(f'\n[{j}] {type(m).__name__}')
    if hasattr(m, 'content') and m.content:
        print(f'    {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        for tc in m.tool_calls:
            print(f'    → {tc.get("name")}({tc.get("args")})')

Failure recovery trace:

[0] HumanMessage
    Fetch this JD and tell me the must-have skills: https://this-does-not-exist-99999.example.com/jd

[1] AIMessage
    → jd_fetcher({'url': 'https://this-does-not-exist-99999.example.com/jd'})

[2] ToolMessage
    ERROR: failed to fetch URL — HTTPSConnectionPool(host='this-does-not-exist-99999.example.com', port=443): Max retries exceeded with url: /jd (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7acb88e23a10>: Failed to resolve 'this-does-not-exist-99999.example.com' ([Errn

[3] AIMessage
    [{'type': 'text', 'text': 'I was not able to fetch the JD from the URL provided. Please provide a valid URL.', 'extras': {'signature': 'Cr8CAQw51sd73sL1QiwvJyJOJrdR4WisMYJCKse5rT6PuWezczmCFcE8E25DHf++ZYie/U3jHqlvSbLwTYc0ZSFxPYutTRRGCzH/0Ff8sPJt4d+c4cSI/AhN/KBH5uuJPvTS856qAWcpHttJRQO9+lkOL/8gwYcsIUHd
